# Ordered Submission Notebook: Churn Prediction + Explainable AI

This notebook is intentionally structured for dissertation submission in the correct analytical order:

1. Data loading and profiling
2. Preprocessing and train-test split
3. Baseline + advanced model setup
4. K-Fold cross-validation and GridSearch tuning
5. **All-model comparison on the test set**
6. **Final model selection**
7. **Explainability (XAI) on the selected final model**
8. Business-value outputs (risk ranking and decile lift)

---

## Core Dissertation Goals
1. **Prediction:** Which customers are likely to churn?
2. **Explanation:** Why are they likely to churn?
3. **Insight:** What factors influence churn?
4. **Value:** How can businesses use this information?


### Cell 1 (Methodology): Imports and reproducibility setup

This cell defines all required libraries, output paths, and reproducibility controls. The configuration ensures that analysis outputs are consistently generated and traceable for dissertation reporting.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    matthews_corrcoef, confusion_matrix, roc_curve, precision_recall_curve
)
from sklearn.inspection import permutation_importance

import shap

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
BASE_DIR = Path(".").resolve()
DATA_PATH = BASE_DIR / "Bank Customer Churn Prediction.csv"
FIG_DIR = BASE_DIR / "outputs" / "figures"
TABLE_DIR = BASE_DIR / "outputs" / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

print("Data path:", DATA_PATH)
print("XGBoost available:", XGBOOST_AVAILABLE)
print("This output shows the configured environment and availability of optional advanced packages.")

### Cell 2 (Methodology): Data loading and profiling

This cell loads the dataset, verifies structure, checks missingness and duplicates, and confirms the target variable. This provides transparent documentation of dataset quality prior to model development.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns = [c.strip().replace(" ", "_") for c in df.columns]

possible_targets = ["churn", "Exited", "Churn", "target", "Target"]
target_col = next((c for c in possible_targets if c in df.columns), None)
if target_col is None:
    raise ValueError(f"Target column not found. Available columns: {df.columns.tolist()}")

print("Shape:", df.shape)
print("Target:", target_col)
print("Missing values total:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

display(df.head())
print("This output shows that the dataset structure and quality checks were completed before modelling.")

### Cell 3 (Results): Core EDA visuals

This cell provides compact but essential descriptive visuals required for methodological transparency and early results interpretation.

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(data=df, x=target_col, palette="Set2")
plt.title("Churn Class Distribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_target_distribution.png", dpi=300)
plt.show()
print("This plot shows the class balance between churned and retained customers.")

num_cols_all = [c for c in df.select_dtypes(include=np.number).columns if c != target_col]
if len(num_cols_all) > 0:
    plt.figure(figsize=(10, 8))
    sns.heatmap(df[[*num_cols_all, target_col]].corr(numeric_only=True), cmap="coolwarm", center=0)
    plt.title("Correlation Heatmap")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_ordered_correlation_heatmap.png", dpi=300)
    plt.show()
    print("This plot shows linear association patterns among numeric predictors and the churn target.")

### Cell 4 (Methodology): Preprocessing and split

This cell creates leakage-safe preprocessing pipelines and performs stratified train-test splitting, ensuring fair model comparison.

In [ ]:
X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

for id_col in ["customer_id", "CustomerId", "RowNumber", "Surname"]:
    if id_col in X.columns:
        X = X.drop(columns=[id_col])

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

try:
    ohe_sparse = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    ohe_dense = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe_sparse = OneHotEncoder(handle_unknown="ignore", sparse=True)
    ohe_dense = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor_sparse = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", ohe_sparse)]), categorical_cols),
])

preprocessor_dense = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", ohe_dense)]), categorical_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Numeric cols:", len(numeric_cols), "| Categorical cols:", len(categorical_cols))
print("This output shows that stratified partitioning and preprocessing design were completed for robust model development.")

### Cell 5 (Methodology): Define baseline and advanced models

This cell specifies baseline and advanced classifiers, including neural and boosting models, to provide broad comparative coverage.

In [ ]:
model_specs = {
    "Logistic Regression": (preprocessor_sparse, LogisticRegression(max_iter=1200, class_weight="balanced", random_state=RANDOM_STATE)),
    "Random Forest": (preprocessor_sparse, RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)),
    "SVM (RBF)": (preprocessor_sparse, SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=RANDOM_STATE)),
    "HistGradientBoosting": (preprocessor_dense, HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
    "MLP Neural Network": (preprocessor_dense, MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=450, random_state=RANDOM_STATE)),
}
if XGBOOST_AVAILABLE:
    model_specs["XGBoost"] = (
        preprocessor_dense,
        XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="binary:logistic",
            eval_metric="auc",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    )

print("Models configured:", list(model_specs.keys()))
print("This output shows the full model set to be compared before final model selection.")

### Cell 6 (Methodology): K-Fold cross-validation (all models)

This cell estimates model generalisation under Stratified 5-Fold cross-validation using a multi-metric framework.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}

cv_rows = []
for name, (prep, model) in model_specs.items():
    pipe = Pipeline([("preprocessor", prep), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1, error_score="raise")
    cv_rows.append({
        "Model": name,
        "CV_Accuracy": scores["test_accuracy"].mean(),
        "CV_Precision": scores["test_precision"].mean(),
        "CV_Recall": scores["test_recall"].mean(),
        "CV_F1": scores["test_f1"].mean(),
        "CV_ROC_AUC": scores["test_roc_auc"].mean(),
        "CV_PR_AUC": scores["test_pr_auc"].mean(),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("CV_F1", ascending=False)
cv_df.to_csv(TABLE_DIR / "table_ordered_cv_all_models.csv", index=False)
display(cv_df)
print("This output shows cross-validated comparative performance prior to final hold-out testing.")

### Cell 7 (Methodology): Hyperparameter tuning (GridSearchCV)

This cell applies GridSearchCV to key advanced models (MLP+PCA and optional XGBoost) to improve performance through systematic optimisation.

In [ ]:
# PCA variance probe for MLP+PCA grid
X_train_dense = preprocessor_dense.fit_transform(X_train)
max_comp = min(50, X_train_dense.shape[1])
pca_probe = PCA(n_components=max_comp, random_state=RANDOM_STATE).fit(X_train_dense)
cum_var = np.cumsum(pca_probe.explained_variance_ratio_)
n95 = int(np.argmax(cum_var >= 0.95) + 1) if np.any(cum_var >= 0.95) else max_comp

plt.figure(figsize=(8,5))
plt.plot(np.arange(1, max_comp+1), cum_var, marker="o")
plt.axhline(0.95, linestyle="--", color="red")
plt.axvline(n95, linestyle="--", color="black", label=f"n={n95}")
plt.title("PCA Cumulative Explained Variance")
plt.xlabel("Components")
plt.ylabel("Cumulative explained variance")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_pca_variance.png", dpi=300)
plt.show()
print("This plot shows how many principal components are required to retain most information content.")

grid_pipelines = {}
grid_rows = []

mlp_grid_pipe = Pipeline([
    ("preprocessor", preprocessor_dense),
    ("pca", PCA(n_components=n95, random_state=RANDOM_STATE)),
    ("model", MLPClassifier(max_iter=500, random_state=RANDOM_STATE))
])
mlp_grid = {
    "model__hidden_layer_sizes": [(64,), (64, 32), (128, 64)],
    "model__alpha": [1e-4, 1e-3],
    "model__learning_rate_init": [1e-3, 5e-4],
}
mlp_search = GridSearchCV(mlp_grid_pipe, mlp_grid, scoring="f1", cv=cv, n_jobs=-1)
mlp_search.fit(X_train, y_train)
grid_pipelines["MLP+PCA (Grid)"] = mlp_search.best_estimator_
grid_rows.append({"Model": "MLP+PCA (Grid)", "BestCV_F1": mlp_search.best_score_, "BestParams": str(mlp_search.best_params_)})

if XGBOOST_AVAILABLE:
    xgb_grid_pipe = Pipeline([
        ("preprocessor", preprocessor_dense),
        ("model", XGBClassifier(objective="binary:logistic", eval_metric="auc", random_state=RANDOM_STATE, n_jobs=-1))
    ])
    xgb_grid = {
        "model__n_estimators": [300, 500],
        "model__learning_rate": [0.03, 0.05],
        "model__max_depth": [3, 4],
    }
    xgb_search = GridSearchCV(xgb_grid_pipe, xgb_grid, scoring="f1", cv=cv, n_jobs=-1)
    xgb_search.fit(X_train, y_train)
    grid_pipelines["XGBoost (Grid)"] = xgb_search.best_estimator_
    grid_rows.append({"Model": "XGBoost (Grid)", "BestCV_F1": xgb_search.best_score_, "BestParams": str(xgb_search.best_params_)})

grid_df = pd.DataFrame(grid_rows).sort_values("BestCV_F1", ascending=False)
grid_df.to_csv(TABLE_DIR / "table_ordered_gridsearch_summary.csv", index=False)
display(grid_df)
print("This output shows the optimal hyperparameter configurations identified by GridSearchCV.")

### Cell 8 (Results): Final all-model comparison on test set

This is the key ordered step: every model (base + advanced + tuned) is evaluated first, then ranked. Only after this table is produced is final model selection performed.

In [ ]:
def eval_model(name, pipe, X_test, y_test):
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    m = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, proba),
        "PR_AUC": average_precision_score(y_test, proba),
        "Brier": brier_score_loss(y_test, proba),
        "MCC": matthews_corrcoef(y_test, pred),
    }
    fpr, tpr, _ = roc_curve(y_test, proba)
    prec, rec, _ = precision_recall_curve(y_test, proba)
    cm = confusion_matrix(y_test, pred)
    return m, fpr, tpr, rec, prec, cm, proba

all_pipelines = {}
for name, (prep, model) in model_specs.items():
    pipe = Pipeline([("preprocessor", prep), ("model", model)])
    pipe.fit(X_train, y_train)
    all_pipelines[name] = pipe
all_pipelines.update(grid_pipelines)

rows, roc_map, pr_map, cm_map, proba_map = [], {}, {}, {}, {}
for name, pipe in all_pipelines.items():
    m, fpr, tpr, rec, prec, cm, proba = eval_model(name, pipe, X_test, y_test)
    rows.append(m)
    roc_map[name] = (fpr, tpr, m["ROC_AUC"])
    pr_map[name] = (rec, prec, m["PR_AUC"])
    cm_map[name] = cm
    proba_map[name] = proba

final_compare_df = pd.DataFrame(rows).sort_values(["F1", "ROC_AUC", "PR_AUC"], ascending=[False, False, False])
final_compare_df.to_csv(TABLE_DIR / "table_ordered_final_model_comparison.csv", index=False)
display(final_compare_df)
print("This output shows the complete model comparison table that precedes final model selection.")

### Cell 9 (Results): Final model selection and comparative plots

This cell selects the best-performing model from the complete comparison table and visualises comparative ROC/PR performance.

In [ ]:
FINAL_MODEL_NAME = final_compare_df.iloc[0]["Model"]
FINAL_MODEL_PIPE = all_pipelines[FINAL_MODEL_NAME]
FINAL_PROBA = proba_map[FINAL_MODEL_NAME]

print("Final selected model:", FINAL_MODEL_NAME)
print("This output shows the final selected model after complete all-model comparison.")

# ROC top models
plt.figure(figsize=(8,6))
for m in final_compare_df.head(min(6, len(final_compare_df)))["Model"]:
    fpr, tpr, auc = roc_map[m]
    plt.plot(fpr, tpr, label=f"{m} (AUC={auc:.3f})")
plt.plot([0,1], [0,1], "k--")
plt.title("ROC Curves (Top Models)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_roc_top_models.png", dpi=300)
plt.show()
print("This plot shows discrimination differences among top candidate models.")

# PR top models
plt.figure(figsize=(8,6))
for m in final_compare_df.head(min(6, len(final_compare_df)))["Model"]:
    rec, prec, ap = pr_map[m]
    plt.plot(rec, prec, label=f"{m} (AP={ap:.3f})")
plt.title("Precision-Recall Curves (Top Models)")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_pr_top_models.png", dpi=300)
plt.show()
print("This plot shows precision-recall trade-offs, which are critical in imbalanced churn prediction.")

### Cell 10 (Results): Explainability on selected final model only

This cell performs permutation importance and SHAP analysis on the selected final model workflow to maintain interpretive consistency.

In [ ]:
# Permutation importance
perm = permutation_importance(FINAL_MODEL_PIPE, X_test, y_test, scoring="f1", n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)
perm_df.to_csv(TABLE_DIR / "table_ordered_final_permutation_importance.csv", index=False)
display(perm_df.head(20))
print("This output shows the most influential predictors under model-agnostic permutation analysis for the final selected model.")

# SHAP (tree-preferred fallback strategy)
def _is_tree_name(name: str) -> bool:
    n = name.lower()
    return any(k in n for k in ["forest", "boost", "xgboost", "tree", "hist"])

shap_pipe = FINAL_MODEL_PIPE
shap_name = FINAL_MODEL_NAME
if not _is_tree_name(FINAL_MODEL_NAME):
    for candidate in final_compare_df["Model"]:
        if _is_tree_name(candidate):
            shap_pipe = all_pipelines[candidate]
            shap_name = candidate
            break

print("SHAP model used:", shap_name)

shap_pre = shap_pipe.named_steps["preprocessor"]
shap_model = shap_pipe.named_steps["model"]
shap_features = shap_pre.get_feature_names_out()
X_test_shap = shap_pre.transform(X_test)
if hasattr(X_test_shap, "toarray"):
    X_test_shap = X_test_shap.toarray()

n_sample = min(400, X_test_shap.shape[0])
idx = np.random.RandomState(RANDOM_STATE).choice(X_test_shap.shape[0], n_sample, replace=False)
X_shap = X_test_shap[idx]

explainer = shap.TreeExplainer(shap_model)
shap_values = explainer.shap_values(X_shap)
if isinstance(shap_values, list) and len(shap_values) == 2:
    shap_matrix = shap_values[1]
elif isinstance(shap_values, np.ndarray) and shap_values.ndim == 3:
    shap_matrix = shap_values[:, :, 1]
else:
    shap_matrix = shap_values

shap_df = pd.DataFrame({"feature": shap_features, "mean_abs_shap": np.abs(shap_matrix).mean(axis=0)}).sort_values("mean_abs_shap", ascending=False)
shap_df.to_csv(TABLE_DIR / "table_ordered_final_shap_global.csv", index=False)
display(shap_df.head(20))
print("This output shows the final global SHAP ranking of churn drivers.")

plt.figure()
shap.summary_plot(shap_matrix, X_shap, feature_names=shap_features, plot_type="bar", max_display=20, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_final_shap_bar.png", dpi=300, bbox_inches="tight")
plt.show()
print("This plot shows the relative contribution magnitude of top features in the final explainability workflow.")

plt.figure()
shap.summary_plot(shap_matrix, X_shap, feature_names=shap_features, max_display=20, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_final_shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()
print("This plot shows the distribution and direction of feature effects on churn predictions.")

### Cell 11 (Results + Value): Final business targeting outputs

This cell produces actionable outputs (top-risk customers and decile lift) from the selected model, linking predictive modelling to operational value.

In [ ]:
final_pred = X_test.copy()
final_pred["true_churn"] = y_test.values
final_pred["predicted_probability"] = FINAL_PROBA

top100 = final_pred.sort_values("predicted_probability", ascending=False).head(100)
top100.to_csv(TABLE_DIR / "table_ordered_final_top100_likely_churners.csv", index=False)

decile = final_pred.copy()
decile["decile"] = pd.qcut(decile["predicted_probability"], q=10, labels=False, duplicates="drop")
decile["decile"] = decile["decile"].max() - decile["decile"] + 1

lift = decile.groupby("decile").agg(
    customers=("true_churn", "size"),
    churners=("true_churn", "sum"),
    avg_probability=("predicted_probability", "mean")
).reset_index().sort_values("decile")
lift["churn_rate"] = lift["churners"] / lift["customers"]
lift["lift_vs_base"] = lift["churn_rate"] / final_pred["true_churn"].mean()
lift.to_csv(TABLE_DIR / "table_ordered_final_decile_lift.csv", index=False)

display(lift)
print("This output shows how strongly the final model concentrates churners into higher-risk deciles.")

plt.figure(figsize=(8,4))
sns.barplot(data=lift, x="decile", y="lift_vs_base", palette="viridis")
plt.axhline(1.0, color="black", linestyle="--", linewidth=1)
plt.title(f"Final Decile Lift ({FINAL_MODEL_NAME})")
plt.xlabel("Risk decile (1 = highest risk)")
plt.ylabel("Lift vs base churn rate")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_final_decile_lift.png", dpi=300)
plt.show()
print("This plot shows the practical targeting value of the selected model for retention intervention design.")

## Dissertation chapter mapping (direct use)

- **Methodology:** Cells 1, 2, 4, 5, 6, 7
- **Results:** Cells 3, 8, 9, 10, 11

For final submission, use this notebook as the primary ordered analysis source.

## Comprehensive Extension (Full Analysis Depth)

This extension incorporates the broader analytical depth from the exploratory notebook while preserving the correct submission order. It adds:

- Extended data diagnostics (audit, outliers, class imbalance)
- Deeper EDA and interaction comparisons
- Additional model diagnostics (confusion matrices, calibration, threshold-cost, learning curves)
- Deeper XAI (dependence and local case explanations)
- Subgroup and business action analysis

This is intended to preserve analytical richness without compromising methodological sequence.

### Cell 12 (Methodology): Extended data audit and diagnostics

This cell provides an expanded audit of feature quality and distribution properties, supporting a transparent and reproducible data preparation narrative.

In [ ]:
audit_df = pd.DataFrame({
    "feature": df.columns,
    "dtype": [str(df[c].dtype) for c in df.columns],
    "missing_count": [df[c].isna().sum() for c in df.columns],
    "missing_percent": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
})
audit_df = audit_df.sort_values(["missing_percent", "n_unique"], ascending=[False, False])
audit_df.to_csv(TABLE_DIR / "table_ordered_feature_audit_extended.csv", index=False)
display(audit_df)
print("This output shows a complete feature-level data quality audit used to justify preprocessing decisions.")

class_dist = y.value_counts().sort_index().rename_axis("class").reset_index(name="count")
class_dist["percent"] = (class_dist["count"] / len(y) * 100).round(2)
class_dist.to_csv(TABLE_DIR / "table_ordered_class_distribution_extended.csv", index=False)
display(class_dist)
print("This output shows class imbalance magnitude and supports metric choice beyond accuracy.")

# Outlier scan (IQR)
def iqr_rate(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return 0.0
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((s < low) | (s > high)).mean()

outlier_df = pd.DataFrame({
    "feature": numeric_cols,
    "iqr_outlier_rate": [iqr_rate(df[c].dropna()) for c in numeric_cols]
}).sort_values("iqr_outlier_rate", ascending=False)
outlier_df.to_csv(TABLE_DIR / "table_ordered_outlier_rates.csv", index=False)
display(outlier_df.head(20))
print("This output shows the relative outlier burden by numeric feature.")

### Cell 13 (Results): Deeper EDA comparisons and interaction visuals

This cell extends descriptive analysis to include group-level churn comparisons and interaction heatmaps, enabling richer behavioural interpretation.

In [ ]:
# Categorical churn-rate comparisons
for c in [col for col in ["country", "gender", "active_member", "credit_card", "products_number"] if col in df.columns]:
    g = df.groupby(c)[target_col].mean().reset_index().sort_values(target_col, ascending=False)
    g.to_csv(TABLE_DIR / f"table_ordered_churn_by_{c}.csv", index=False)

    plt.figure(figsize=(8,4))
    sns.barplot(data=g, x=c, y=target_col, palette="crest")
    plt.title(f"Churn Rate by {c}")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"fig_ordered_churn_by_{c}.png", dpi=300)
    plt.show()
    print(f"This plot shows how churn incidence differs across groups of {c}.")

# Age bands if available
if "age" in df.columns:
    age_bins = [0, 25, 35, 45, 55, 65, 100]
    age_labels = ["<=25", "26-35", "36-45", "46-55", "56-65", "65+"]
    temp = df.copy()
    temp["age_band"] = pd.cut(temp["age"], bins=age_bins, labels=age_labels, include_lowest=True)
    age_tab = temp.groupby("age_band")[target_col].mean().reset_index()
    age_tab.to_csv(TABLE_DIR / "table_ordered_churn_by_age_band.csv", index=False)

    plt.figure(figsize=(8,4))
    sns.barplot(data=age_tab, x="age_band", y=target_col, palette="magma")
    plt.title("Churn Rate by Age Band")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_ordered_churn_by_age_band.png", dpi=300)
    plt.show()
    print("This plot shows how churn risk changes across age cohorts.")

# Interaction heatmap country x active_member
if all(c in df.columns for c in ["country", "active_member"]):
    piv = df.pivot_table(values=target_col, index="country", columns="active_member", aggfunc="mean")
    piv.to_csv(TABLE_DIR / "table_ordered_interaction_country_active.csv")

    plt.figure(figsize=(7,4))
    sns.heatmap(piv, annot=True, fmt=".3f", cmap="YlOrRd")
    plt.title("Churn Interaction: Country x Active Member")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig_ordered_interaction_country_active.png", dpi=300)
    plt.show()
    print("This plot shows interaction effects between geography and account activity on churn.")

### Cell 14 (Results): Additional model diagnostics after final selection

This cell extends performance interpretation using confusion matrices, calibration, learning curve, and threshold-cost diagnostics for the selected final model.

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.model_selection import learning_curve

# Confusion matrix
final_pred_label = (FINAL_PROBA >= 0.5).astype(int)
cm = confusion_matrix(y_test, final_pred_label)

plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title(f"Confusion Matrix: {FINAL_MODEL_NAME}")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_final_confusion_matrix.png", dpi=300)
plt.show()
print("This plot shows the classification error structure of the selected final model.")

# Calibration
prob_true, prob_pred = calibration_curve(y_test, FINAL_PROBA, n_bins=10, strategy="quantile")
plt.figure(figsize=(6,6))
plt.plot(prob_pred, prob_true, marker="o", label="Model")
plt.plot([0,1], [0,1], "k--", label="Perfect calibration")
plt.title("Calibration Curve (Final Model)")
plt.xlabel("Predicted probability")
plt.ylabel("Observed frequency")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_final_calibration_curve.png", dpi=300)
plt.show()
print("This plot shows how well predicted probabilities align with observed churn frequencies.")

# Learning curve
sizes, train_scores, val_scores = learning_curve(
    FINAL_MODEL_PIPE, X_train, y_train, cv=5, scoring="f1", n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 6)
)

plt.figure(figsize=(8,5))
plt.plot(sizes, train_scores.mean(axis=1), marker="o", label="Train F1")
plt.plot(sizes, val_scores.mean(axis=1), marker="o", label="Validation F1")
plt.title(f"Learning Curve: {FINAL_MODEL_NAME}")
plt.xlabel("Training samples")
plt.ylabel("F1 score")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_final_learning_curve.png", dpi=300)
plt.show()
print("This plot shows model learning behaviour and potential underfitting/overfitting patterns.")

# Threshold and business cost
thr_grid = np.arange(0.05, 0.96, 0.01)
COST_FN, COST_FP = 5, 1
thr_rows = []
for t in thr_grid:
    p = (FINAL_PROBA >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, p).ravel()
    thr_rows.append({
        "threshold": t,
        "precision": precision_score(y_test, p, zero_division=0),
        "recall": recall_score(y_test, p, zero_division=0),
        "f1": f1_score(y_test, p, zero_division=0),
        "business_cost": fn*COST_FN + fp*COST_FP,
    })
thr_df = pd.DataFrame(thr_rows)
thr_df.to_csv(TABLE_DIR / "table_ordered_final_threshold_cost.csv", index=False)

best_f1_thr = thr_df.loc[thr_df["f1"].idxmax(), "threshold"]
best_cost_thr = thr_df.loc[thr_df["business_cost"].idxmin(), "threshold"]

plt.figure(figsize=(8,5))
plt.plot(thr_df["threshold"], thr_df["precision"], label="Precision")
plt.plot(thr_df["threshold"], thr_df["recall"], label="Recall")
plt.plot(thr_df["threshold"], thr_df["f1"], label="F1", linewidth=2)
plt.axvline(best_f1_thr, linestyle="--", color="black", label=f"Best F1={best_f1_thr:.2f}")
plt.title("Threshold Sensitivity (Final Model)")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_final_threshold_sensitivity.png", dpi=300)
plt.show()
print("This plot shows how model performance metrics change as the decision threshold varies.")

plt.figure(figsize=(8,5))
plt.plot(thr_df["threshold"], thr_df["business_cost"], color="#C44E52", linewidth=2)
plt.axvline(best_cost_thr, linestyle="--", color="black", label=f"Min Cost={best_cost_thr:.2f}")
plt.title("Threshold vs Business Cost")
plt.xlabel("Threshold")
plt.ylabel("Business cost")
plt.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_final_threshold_business_cost.png", dpi=300)
plt.show()
print("This plot shows the cost-optimal operating threshold under the defined false negative/false positive cost assumptions.")

### Cell 15 (Results): Deeper explainability (dependence + local cases)

This cell extends explainability by adding SHAP dependence plots and local case-level attributions, improving interpretive depth for dissertation discussion.

In [ ]:
# Top SHAP features for dependence
mean_abs = np.abs(shap_matrix).mean(axis=0)
rank_idx = np.argsort(mean_abs)[::-1]
top_feats = [shap_features[i] for i in rank_idx[:3]]

for feat in top_feats:
    plt.figure()
    shap.dependence_plot(feat, shap_matrix, X_shap, feature_names=shap_features, show=False)
    plt.tight_layout()
    safe = feat.replace("__", "_").replace("/", "_")
    plt.savefig(FIG_DIR / f"fig_ordered_final_shap_dependence_{safe}.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"This plot shows how {feat} influences churn predictions across its value range.")

# Local SHAP tables for sample cases
for case_idx in [0, min(1, len(X_shap)-1), min(2, len(X_shap)-1)]:
    local_df = pd.DataFrame({
        "feature": shap_features,
        "shap_value": shap_matrix[case_idx],
        "abs_shap": np.abs(shap_matrix[case_idx]),
        "feature_value": X_shap[case_idx],
    }).sort_values("abs_shap", ascending=False)

    local_df.head(15).to_csv(TABLE_DIR / f"table_ordered_final_local_shap_case_{case_idx}.csv", index=False)
    display(local_df.head(10))
    print("This output shows the dominant feature-level drivers for an individual customer prediction.")

### Cell 16 (Results + Discussion): Subgroup analysis and fairness-style diagnostics

This cell compares observed churn and predicted risk across key groups to evaluate segment-level consistency and transparency.

In [ ]:
sub_df = X_test.copy()
sub_df["true_churn"] = y_test.values
sub_df["pred_proba"] = FINAL_PROBA

for g in [c for c in ["country", "gender", "active_member", "products_number"] if c in sub_df.columns]:
    tab = sub_df.groupby(g).agg(
        n=("true_churn", "size"),
        observed_churn=("true_churn", "mean"),
        avg_predicted_risk=("pred_proba", "mean")
    ).reset_index().sort_values("observed_churn", ascending=False)

    tab.to_csv(TABLE_DIR / f"table_ordered_final_subgroup_{g}.csv", index=False)
    display(tab)
    print("This output shows subgroup-level observed and predicted churn alignment.")

    fig, axes = plt.subplots(1,2, figsize=(12,4))
    sns.barplot(data=tab, x=g, y="observed_churn", ax=axes[0], palette="Reds")
    axes[0].set_title(f"Observed churn by {g}")
    axes[0].tick_params(axis="x", rotation=30)

    sns.barplot(data=tab, x=g, y="avg_predicted_risk", ax=axes[1], palette="Blues")
    axes[1].set_title(f"Predicted risk by {g}")
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.savefig(FIG_DIR / f"fig_ordered_final_subgroup_{g}.png", dpi=300)
    plt.show()
    print("This plot shows whether model risk scores are consistent with observed churn behaviour across groups.")

### Cell 17 (Value): Action-priority matrix and intervention planning

This cell converts final model outputs into practical action tiers for customer retention planning.

In [ ]:
action_df = final_pred.copy()

conditions = [
    action_df["predicted_probability"] >= 0.80,
    (action_df["predicted_probability"] >= 0.60) & (action_df["predicted_probability"] < 0.80),
    (action_df["predicted_probability"] >= 0.40) & (action_df["predicted_probability"] < 0.60),
]
labels = ["Immediate", "High", "Medium"]
action_df["intervention_priority"] = np.select(conditions, labels, default="Monitor")

if "active_member" in action_df.columns:
    action_df["engagement_action"] = np.where(action_df["active_member"] == 0, "Engagement Campaign", "Retention Offer")
else:
    action_df["engagement_action"] = "Retention Offer"

if "products_number" in action_df.columns:
    action_df["product_action"] = np.where(action_df["products_number"] <= 1, "Cross-sell Bundle", "Loyalty Benefits")
else:
    action_df["product_action"] = "Loyalty Benefits"

priority_tab = action_df.groupby("intervention_priority").agg(
    customers=("true_churn", "size"),
    observed_churn=("true_churn", "mean"),
    avg_risk=("predicted_probability", "mean")
).reset_index().sort_values("avg_risk", ascending=False)

priority_tab.to_csv(TABLE_DIR / "table_ordered_final_action_priority_summary.csv", index=False)
action_df.to_csv(TABLE_DIR / "table_ordered_final_action_priority_customer_level.csv", index=False)

display(priority_tab)
print("This output shows operational prioritisation groups derived from final model risk scores.")

plt.figure(figsize=(8,4))
sns.barplot(data=priority_tab, x="intervention_priority", y="customers", palette="Purples")
plt.title("Customers by Intervention Priority")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_final_customers_by_priority.png", dpi=300)
plt.show()
print("This plot shows the population size assigned to each intervention tier.")

plt.figure(figsize=(8,4))
sns.barplot(data=priority_tab, x="intervention_priority", y="observed_churn", palette="OrRd")
plt.title("Observed Churn by Intervention Priority")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_ordered_final_churn_by_priority.png", dpi=300)
plt.show()
print("This plot shows that higher intervention-priority groups correspond to higher empirical churn intensity.")

## Final submission guidance

This ordered notebook now combines:
- full analytical depth from the exploratory notebook, and
- strict methodological sequence for dissertation submission.

### Recommended chapter usage
- **Methodology:** Cells 1, 2, 4, 5, 6, 7, 12
- **Results:** Cells 3, 8, 9, 10, 11, 13, 14, 15, 16, 17

Use generated files in `outputs/figures/` and `outputs/tables/` for figure/table insertion and citation.